# Config

> The tunnel route, study area, modelling assumptions and file paths shared by every notebook.

In [ ]:
#| default_exp config

In [ ]:
#| export
import os
from pathlib import Path

import matplotlib as mpl
import pyproj
from shapely.geometry import LineString, Point

## Paths

Downloads are cached in `data/raw/`, intermediate outputs go to `data/derived/` and blog
charts to `figures/`. Two inputs can't be downloaded without an account and are read from
environment variables:

- `NPS_DIR`: HM Land Registry National Polygon Service shapefiles (`LR_POLY_FULL_*.shp`).
- `COD_DIR`: the CCOD and OCOD bulk files (`CCOD_FULL_*.zip`, `OCOD_FULL_*.zip`), defaulting to `data/`.

In [ ]:
#| export
ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
DERIVED = ROOT / "data" / "derived"
FIGURES = ROOT / "figures"
for _d in (RAW, DERIVED, FIGURES):
    _d.mkdir(parents=True, exist_ok=True)

NPS_DIR = Path(os.environ.get("NPS_DIR", ROOT / "data" / "nps"))
COD_DIR = Path(os.environ.get("COD_DIR", ROOT / "data"))

BNG = "EPSG:27700"
WGS84 = "EPSG:4326"

## The route

Durand's Wharf park on Rotherhithe Street to the West India Docks impound lock on Westferry Road,
about 460 m in a straight line and about 50 m south of the Jubilee line tunnels.

In [ ]:
#| export
# (lon, lat)
SOUTH_PORTAL = (-0.032771, 51.501797)  # Durand's Wharf park, Rotherhithe
NORTH_PORTAL = (-0.026450, 51.503040)  # impound lock land, Westferry Road
CANARY_WHARF = (-0.0195700, 51.5049700)  # One Canada Square: stand-in for "the jobs"

_to_bng = pyproj.Transformer.from_crs(WGS84, BNG, always_xy=True)


def to_bng(lonlat: tuple[float, float]) -> tuple[float, float]:
    "Convert a (lon, lat) pair to British National Grid (x, y)."
    return _to_bng.transform(*lonlat)


def tunnel_line() -> LineString:
    "Straight line between the two portals, in BNG."
    return LineString([to_bng(SOUTH_PORTAL), to_bng(NORTH_PORTAL)])

In [ ]:
print(f"Portal to portal: {tunnel_line().length:.0f} m")

Portal to portal: 460 m


## Study area and catchment

In [ ]:
#| export
# (west, south, east, north): Rotherhithe, Bermondsey, Deptford, Isle of Dogs, Wapping
STUDY_BBOX = (-0.085, 51.470, 0.012, 51.522)

# Land readjustment area: this radius around the south portal, Rotherhithe side only
CATCHMENT_M = 750


def catchment() -> Point:
    "The readjustment circle around the south portal, in BNG (not yet clipped to the river)."
    return Point(to_bng(SOUTH_PORTAL)).buffer(CATCHMENT_M)

## Tunnel assumptions

Edit and re-run the notebooks that follow.

In [ ]:
#| export
TUNNEL_DETOUR = 1.10  # bored length / straight-line portal distance
PORTAL_DELAY_MIN = 1.5  # per portal: lift wait + ride (or escalator/ramp)
WALK_KMH = 4.8
CYCLE_KMH = 15.0
PRICE_YEAR = 2025  # all money in these prices


def tunnel_length_m() -> float:
    "Assumed bored length."
    return tunnel_line().length * TUNNEL_DETOUR

In [ ]:
print(f"Assumed tunnel length: {tunnel_length_m():.0f} m")

Assumed tunnel length: 506 m


## Chart style

One palette and style for every blog figure.

In [ ]:
#| export
COLOURS = {
    "tunnel": "#e8673a",
    "southwark": "#2b78d6",
    "housing": "#ee6a37",
    "other": "#1db07f",
    "rest": "#b8b8b0",
    "private": "#dcdcd6",
    "water": "#d6e4ef",
    "land": "#f0efea",
    "park": "#cfe3c8",
    "jubilee": "#7b868c",
    "text": "#555555",
}
BACKGROUND = "#fbfbf9"


def use_style() -> None:
    "Apply the blog chart style to matplotlib."
    mpl.rcParams.update(
        {
            "figure.facecolor": BACKGROUND,
            "axes.facecolor": BACKGROUND,
            "savefig.facecolor": BACKGROUND,
            "savefig.dpi": 150,
            "savefig.bbox": "tight",
            "font.size": 11,
            "axes.titlesize": 15,
            "axes.titleweight": "bold",
            "axes.titlelocation": "left",
            "axes.spines.top": False,
            "axes.spines.right": False,
            "axes.edgecolor": "#aaaaaa",
            "axes.labelcolor": COLOURS["text"],
            "xtick.color": COLOURS["text"],
            "ytick.color": COLOURS["text"],
            "legend.frameon": False,
        }
    )